# Model Validation Notebook

In [ ]:
import sys
from pathlib import Path
import torch
import numpy as np
from torch.utils.data import DataLoader
from torchvision import transforms
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

def find_project_root(start_path: Path, required_dir: str = "src") -> Path:
    current = start_path.resolve()
    while True:
        if (current / required_dir).exists():
            return current
        if current.parent == current:
            raise FileNotFoundError(f"Cannot find project root containing '{required_dir}'")
        current = current.parent

PROJECT_ROOT = find_project_root(Path.cwd(), required_dir="src")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.datasets.multimodal_raw_dataset import MultiModalRawDataset, multimodal_raw_collate_fn
from src.models.multimodal.multimodal_text_guided_pvd_infonce_supcon import MultimodalTextGuidedPVDInfoNCESupCon
from src.models.losses.wrapper_loss import InfoNCESupConLoss

print("PROJECT_ROOT:", PROJECT_ROOT)

## 1. Tải Checkpoint & Khôi phục Cấu hình (Config)

In [ ]:
# Thư mục lưu mô hình của bạn
ARCHIVE_DIR = "archive/text_guided_infonce_supcon_multihead_attention_pvd"
CHECKPOINT_NAME = "best_model.pt"

ckpt_path = PROJECT_ROOT / ARCHIVE_DIR / CHECKPOINT_NAME
print(f"Loading checkpoint from: {ckpt_path}")

if not ckpt_path.exists():
    raise FileNotFoundError(f"Không tìm thấy file trọng số tại: {ckpt_path}")

# Load checkpoint lên CPU trước để tránh đầy VRAM
checkpoint = torch.load(ckpt_path, map_location="cpu")
config = checkpoint["config"]

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device used for evaluation: {device}")
print(f"Model was trained up to epoch: {checkpoint.get('epoch', 'Unknown')}")
print(f"Best Validation Metric (Accuracy) during training: {checkpoint.get('best_metric', 'Unknown')}")

## 2. Khởi tạo Dataset & DataLoader

In [ ]:
def build_val_transform(img_size: int):
    return transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

val_transform = build_val_transform(config["data"]["img_size"])

# Chỉ định đường dẫn tới tập Validation (hoặc thay bằng tập Test nếu có)
val_image_root = PROJECT_ROOT / config["data"]["val_image_root"]
val_caption_root = PROJECT_ROOT / config["data"]["val_caption_root"]

val_dataset = MultiModalRawDataset(
    image_root=val_image_root,
    caption_root=val_caption_root,
    transform=val_transform,
    use_depth_suppressed=config["data"].get("use_depth_suppressed", False),
    strict_caption_match=config["data"].get("strict_caption_match", True),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config["training"].get("batch_size", 8),
    shuffle=False,
    num_workers=config["training"].get("num_workers", 4),
    pin_memory=torch.cuda.is_available(),
    collate_fn=multimodal_raw_collate_fn,
)

print(f"Validation samples: {len(val_dataset)}")

## 3. Nạp Model & Trọng số

In [ ]:
model = MultimodalTextGuidedPVDInfoNCESupCon(
    num_classes=config["model"]["num_classes"],
    clip_model_name=config["model"]["clip_model_name"],
    clip_pretrained=config["model"]["clip_pretrained"],
    text_input_dim=config["model"]["text_input_dim"],
    image_input_dim=config["model"]["image_input_dim"],
    proj_dim=config["model"]["proj_dim"],
    proj_hidden_dim=config["model"]["proj_hidden_dim"],
    pvd_hidden_dim=config["model"]["pvd_hidden_dim"],
    cls_hidden_dim=config["model"]["cls_hidden_dim"],
    dropout=config["model"]["dropout"],
    normalize_projection=config["model"]["normalize_projection"],
    device=device,
).to(device)

# Sử dụng strict=False phòng trường hợp có tinh chỉnh nhỏ về module
model.load_state_dict(checkpoint["model_state_dict"], strict=False)
model.eval()  # Chuyển sang chế độ đánh giá
torch.cuda.empty_cache()

criterion = InfoNCESupConLoss(
    ce_weight=config["loss"]["ce_weight"],
    itc_weight=config["loss"]["itc_weight"],
    supcon_weight=config["loss"]["supcon_weight"],
    itc_temperature=config["loss"]["itc_temperature"],
    supcon_temperature=config["loss"]["supcon_temperature"],
)

print("Model loaded successfully!")

## 4. Hàm Evaluation

In [ ]:
@torch.no_grad()
def evaluate_model(model, dataloader, criterion, device="cpu"):
    total_samples, total_correct = 0, 0
    running_loss, running_ce, running_itc, running_supcon = 0.0, 0.0, 0.0, 0.0

    all_preds = []
    all_labels = []

    progress_bar = tqdm(dataloader, desc="Evaluating")
    for batch in progress_bar:
        images = batch["image"].to(device)
        texts = batch["text"]
        labels = batch["label"].to(device)

        outputs = model(images, texts)
        loss_dict = criterion(outputs, labels)

        logits = outputs["logits"]
        preds = torch.argmax(logits, dim=1)

        batch_size = labels.size(0)
        total_samples += batch_size
        total_correct += (preds == labels).sum().item()

        running_loss += loss_dict["loss"].item() * batch_size
        running_ce += loss_dict["loss_ce"].item() * batch_size
        running_itc += loss_dict["loss_itc"].item() * batch_size
        running_supcon += loss_dict["loss_supcon"].item() * batch_size

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

        progress_bar.set_postfix({
            "loss": f"{running_loss / max(total_samples, 1):.4f}", 
            "acc": f"{total_correct / max(total_samples, 1):.4f}"
        })

    metrics = {
        "loss": running_loss / total_samples,
        "loss_ce": running_ce / total_samples,
        "loss_itc": running_itc / total_samples,
        "loss_supcon": running_supcon / total_samples,
        "accuracy": total_correct / total_samples
    }
    return metrics, all_preds, all_labels

## 5. Chạy Đánh Giá và Xem Kết Quả

In [ ]:
# Thực thi
metrics, y_pred, y_true = evaluate_model(model, val_loader, criterion, device)

print("\n" + "="*40)
print("VALIDATION METRICS SUMMARY")
print("="*40)
for k, v in metrics.items():
    print(f"{k.upper()}: {v:.4f}")

# Trích xuất tên các classes từ từ điển idx_to_class của dataset
idx_to_class = val_dataset.idx_to_class
target_names = [idx_to_class[i] for i in range(config["model"]["num_classes"])]

# In Báo cáo phân loại chi tiết (Precision, Recall, F1-Score)
print("\n" + "="*60)
print("CLASSIFICATION REPORT")
print("="*60)
print(classification_report(y_true, y_pred, target_names=target_names, digits=4))

# Vẽ biểu đồ Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt="d", cmap='Blues', xticklabels=target_names, yticklabels=target_names)
plt.title(f'Confusion Matrix (Total Acc: {metrics["accuracy"]:.4f})', fontsize=16)
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()
